# nn-module-subclass — ex4: Dropout-style Module that branches on self.training

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `nn-module-subclass`. Running the final beacon cell reports progress against the `PyTorch: nn.Module subclassing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: nn.Module subclassing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nn-module-subclass`** (exercise 4). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nn-module-subclass"
DD_SUBTOPIC = "PyTorch: nn.Module subclassing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## nn.Module subclassing — quick refresher

An `nn.Module` subclass is just a Python class with three contracts:
1. Call `super().__init__()` first when you have state.
2. Implement `forward(self, ...)` — never call `.forward()` directly; use `module(x)`, which routes through `__call__`.
3. The base class also tracks `self.training` (default `True`). `.train()` and `.eval()` flip it for the module AND all children.

**This drill (ex4) vs prior exercises.** ex1 built a stateless square layer; ex2 diagnosed a missing `super().__init__()`; ex3 wrote Linear from scratch. None of them touched the `train`/`eval` mode toggle — the most common reason a `forward` reads `self.<something>` and branches at runtime. Dropout, BatchNorm, and inference-only paths all key off `self.training`.

### Exercise 4 — Dropout-style Module that branches on self.training

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `self.training` flag inside `forward` to branch a Module between a train-time scaling path and an eval-time identity path, exactly as Dropout does.
> Keywords: self.training, train-eval-toggle, dropout-style, forward-branching
> ```

**KCs targeted:** `module-self-training-flag`, `module-subclass-trivial-forward`

Implement `ScalingDropoutLike` — a Module whose forward pass depends on the `self.training` flag set by `.train()`/`.eval()`.

1. Subclass `t.nn.Module`. In `__init__(self, scale)` call `super().__init__()` first; store `self.scale = float(scale)` (a plain Python float, NOT a Parameter — this drill isolates the mode-branching mechanic).
2. In `forward(self, x: Tensor) -> Tensor`:
   - If `self.training` is `True`, return `x * self.scale` (train path — analogous to Dropout's inverse-scaling).
   - Otherwise, return `x` unchanged (eval / inference path).
3. Return an INSTANCE from `ex4_build_scaling_dropout_like(scale)`.

The test toggles `.train()` and `.eval()` and asserts the forward output flips between the two paths. It also asserts that `.eval()` on the parent flips `self.training` to `False` (propagation through `nn.Module.train(mode=False)`).

**No __call__ tricks needed.** `nn.Module.__call__` already routes to `forward` AND respects the `training` flag for you.

In [ ]:
def ex4_build_scaling_dropout_like(scale: float) -> 't.nn.Module':
    """Return a Module whose forward branches on self.training."""
    raise NotImplementedError()


def _test_ex4():
    mod = ex4_build_scaling_dropout_like(0.5)
    assert isinstance(mod, t.nn.Module), f'expected nn.Module, got {type(mod).__name__}'

    # Default state: training=True (nn.Module default).
    assert mod.training is True, f'expected training=True at construction, got {mod.training}'

    x = t.tensor([1.0, 2.0, 4.0, 8.0])

    # Train path: scaled by 0.5.
    mod.train()
    assert mod.training is True
    out_train = mod(x)
    expected_train = x * 0.5
    assert t.allclose(out_train, expected_train), (
        f'train path: got {out_train}, expected {expected_train}'
    )

    # Eval path: identity.
    mod.eval()
    assert mod.training is False, f'.eval() must flip training to False, got {mod.training}'
    out_eval = mod(x)
    assert t.allclose(out_eval, x), f'eval path: got {out_eval}, expected identity {x}'

    # Back to train.
    mod.train()
    assert mod.training is True
    out_train2 = mod(x)
    assert t.allclose(out_train2, expected_train), 'second .train() should re-enable scaling'

    # Different scale → different train output (sanity).
    mod2 = ex4_build_scaling_dropout_like(3.0)
    mod2.train()
    assert t.allclose(mod2(x), x * 3.0)
    mod2.eval()
    assert t.allclose(mod2(x), x)
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
class ScalingDropoutLike(t.nn.Module):
    def __init__(self, scale):
        super().__init__()
        self.scale = float(scale)
    def forward(self, x):
        if self.training:
            return x * self.scale
        return x

def ex4_build_scaling_dropout_like(scale):
    return ScalingDropoutLike(scale)
```

**`self.training` is inherited.** `nn.Module.__init__` sets `self.training = True` for you — that's why a fresh Module is in train mode by default. `.train()` and `.eval()` are also inherited methods that flip the flag recursively across children.

**Why this branching pattern is everywhere.** Dropout multiplies by `1/(1-p)` during training and is the identity at eval. BatchNorm uses the running stats at eval but per-batch stats during training. Any layer that should behave differently for evaluation reads `self.training` inside `forward`.

**Common pitfall.** Forgetting `model.eval()` before a validation loop silently keeps Dropout / BatchNorm in training mode — the classic source of "my model performs worse at inference" bugs.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex4',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()